# Implementing kinase-library code for kinase imputation

In [1]:
from utils import *
import kinase_library as kl

In [2]:
df = pd.read_csv("Experiment/3_hTERT_HME1_mutants_comparison/Data/Processed/Mutant_cell_lines_test.tsv", sep= "\t")

# Remove rows without PTM
df_filtered = df[df.Assigned_Modifications_clean.notna()]
print(f"Original size: {df.shape}")

# Drop rows where site is not detected in the starvation control
df_filtered = df_filtered[df_filtered["WT_raw:abs_EGF_starve_r1"] != 0]
df_filtered = df_filtered[df_filtered["BRAFS151A_raw:abs_EGF_starve_r1"] != 0]
df_filtered = df_filtered[df_filtered["GAB1Y259A_raw:abs_EGF_starve_r1"] != 0]

raw_columns = [element for element in df.columns if 'raw' in element]

print(f"Dataframe after removing sites with no starve detected: {df_filtered.shape}")
replicates = [element for element in raw_columns if "starve" not in element]
# print(f"Raw columns of time points replicates: {replicates}")

replicates = list(set([element.replace("_r1", "").replace("_r2", "") for element in replicates]))

suffixes = ["_r1", "_r2"]

# Drop rows if the phosphosite was not detected in the 2 replicates for any time point and any condition
for condition in replicates:
    mask = (
        (df_filtered[f"{condition}{suffixes[0]}"] == 0) &
        (df_filtered[f"{condition}{suffixes[1]}"] == 0)
    )
    df_filtered = df_filtered.drop(df_filtered[mask].index)

print(f"Dataframe after removing sites where a site is missing in the two replivates: {df_filtered.shape}")

df_filtered = filter_dynamics_extremes_mutants(df = df_filtered,
                                       data_type="log2:FC",
                                       threshold=0.5,
                                       exclude_full = False,
                                       conditions = ["_EGF_"],
                                       cell_lines = ["WT", "BRAFS151A", "GAB1Y259A"],)
print(f"Dataframe after selecting log2:FC > 0.5: {df_filtered.shape}")

Original size: (175089, 136)
Dataframe after removing sites with no starve detected: (13474, 136)
Dataframe after removing sites where a site is missing in the two replivates: (10885, 136)
Dataframe after selecting log2:FC > 0.5: (9561, 136)


In [3]:
df_filtered["site_sequence"] = (df_filtered["Modified_Sequence"].str.replace("S[79.9663]", "s", regex=False)
                    .str.replace("T[79.9663]", "t", regex=False)
                    .str.replace("Y[79.9663]", "y", regex=False)
                    .str.replace("[15.9949]", "", regex=False))


def extract_window(seq, left=6, right=6):
    for i, char in enumerate(seq):
        if char.islower():
            start = max(0, i - left)
            end = min(len(seq), i + right + 1)
            window_seq = seq[start:end]
            left_pad = "_" * max(0, left - i)
            right_pad = "_" * max(0, (i + right + 1) - len(seq))
            return (left_pad + window_seq + right_pad)
    return None

df_filtered["site_sequence"] = df_filtered["site_sequence"].apply(extract_window)

df_filtered

,protein_Id,protein_name,description,Peptide_Sequence,Modified_Sequence,site_start,site_end,Peptide_Length,Charges,Assigned_Modifications,...,WT_log2:FC_EGF_25,BRAFS151A_log2:FC_EGF_starve,BRAFS151A_log2:FC_EGF_2,BRAFS151A_log2:FC_EGF_10,BRAFS151A_log2:FC_EGF_25,GAB1Y259A_log2:FC_EGF_starve,GAB1Y259A_log2:FC_EGF_2,GAB1Y259A_log2:FC_EGF_10,GAB1Y259A_log2:FC_EGF_25,site_sequence
17,Q9NQS7,INCENP,Inner centromere protein,AAAAAAAATMALAAPSSPTPESPTMLTK,AAAAAAAATMALAAPS[79.9663]SPTPES[79.9663]PTMLTK,127,154,28,"2,3","16S(79.9663),22S(79.9663)",...,-0.179979,0.0,0.888322,0.803156,0.130157,0.0,-0.955918,0.016177,0.179395,MALAAPsSPTPEs
38,Q6SPF0,SAMD1,Sterile alpha motif domain-containing protein 1,AAAAAATAPPSPGPAQPGPR,AAAAAATAPPS[79.9663]PGPAQPGPR,151,170,20,2,11S(79.9663),...,0.354369,0.0,-0.962931,-0.290644,-0.431276,0.0,-0.080581,-0.012117,-0.075329,AATAPPsPGPAQP
43,P19338,NCL,Nucleolin,AAAAAPASEDEDDEDDEDDEDDDDDEEDDSEEEAMETTPAK,AAAAAPASEDEDDEDDEDDEDDDDDEEDDS[79.9663]EEEAMET...,177,217,41,3,30S(79.9663),...,0.330201,0.0,-2.574105,-0.605887,-1.345394,0.0,-0.221699,-0.409022,0.028441,DDEEDDsEEEAME
44,P19338,NCL,Nucleolin,AAAAAPASEDEDDEDDEDDEDDDDDEEDDSEEEAMETTPAK,AAAAAPASEDEDDEDDEDDEDDDDDEEDDS[79.9663]EEEAM[1...,177,217,41,3,"30S(79.9663),35M(15.9949)",...,0.527729,0.0,-1.980528,-0.572586,-1.853071,0.0,0.013684,-0.283210,-0.064792,DDEEDDsEEEAME
45,P19338,NCL,Nucleolin,AAAAAPASEDEDDEDDEDDEDDDDDEEDDSEEEAMETTPAK,AAAAAPAS[79.9663]EDEDDEDDEDDEDDDDDEEDDSEEEAMET...,177,217,41,3,8S(79.9663),...,0.545618,0.0,-2.079285,-0.628950,-2.164579,0.0,-0.291065,-0.513799,0.083652,AAAAPAsEDEDDE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175069,P38159,RBMX,"RNA-binding motif protein, X chromosome",VEADRPGK,n[42.0106]VEADRPGK,2,9,8,2,N-term(42.0106),...,0.517346,0.0,-1.805978,-1.245894,-1.831771,0.0,-0.343773,-0.090072,-0.733578,______n[42.01
175071,Q9UBE0,SAE1,SUMO-activating enzyme subunit 1,VEKEEAGGGISEEEAAQYDR,n[42.0106]VEKEEAGGGISEEEAAQYDR,2,21,20,2,N-term(42.0106),...,0.296446,0.0,1.159700,-0.228971,0.361825,0.0,-0.482218,-0.219705,-1.151048,______n[42.01
175082,P55735,SEC13,Protein SEC13 homolog,VSVINTVDTSHEDMIHDAQMDYYGTR,n[42.0106]VSVINTVDTSHEDMIHDAQMDYYGTR,2,27,26,"2,3",N-term(42.0106),...,-0.597662,0.0,-0.829129,-1.267637,-2.648075,0.0,-0.588828,0.145323,-0.892899,______n[42.01
175083,P55735,SEC13,Protein SEC13 homolog,VSVINTVDTSHEDMIHDAQMDYYGTR,n[42.0106]VSVINTVDTSHEDMIHDAQM[15.9949]DYYGTR,2,27,26,3,"N-term(42.0106),20M(15.9949)",...,0.226112,0.0,-1.140735,-0.959473,-1.110505,0.0,-1.318825,-1.071842,-1.446663,______n[42.01


In [5]:
pps = kl.PhosphoProteomics(df_filtered, seq_col="site_sequence", pp=True)

802 entries were omitted due to empty value in the substrates column.
621 entries were omitted due to invalid amino acids or characters.
Use the 'omited_entries' attribute to view dropped enteries due to invalid sequences.


In [6]:
# pps.score(kin_type='ser_thr')
# pps.percentile('ser_thr')
ranked = pps.rank(metric='score', kin_type='ser_thr')
# pps.promiscuity_index(kin_type='ser_thr', metric='percentile', threshold=90)
# pps.predict(kin_type='ser_thr')
ranked

Scoring 8017 ser_thr substrates


,protein_Id,protein_name,description,Peptide_Sequence,Modified_Sequence,site_start,site_end,Peptide_Length,Charges,Assigned_Modifications,...,VRK2,WNK1,WNK2,WNK3,WNK4,YANK2,YANK3,YSK1,YSK4,ZAK
17,Q9NQS7,INCENP,Inner centromere protein,AAAAAAAATMALAAPSSPTPESPTMLTK,AAAAAAAATMALAAPS[79.9663]SPTPES[79.9663]PTMLTK,127,154,28,"2,3","16S(79.9663),22S(79.9663)",...,252,145,202,267,265,208,100,257,106,260
38,Q6SPF0,SAMD1,Sterile alpha motif domain-containing protein 1,AAAAAATAPPSPGPAQPGPR,AAAAAATAPPS[79.9663]PGPAQPGPR,151,170,20,2,11S(79.9663),...,196,158,99,193,205,241,109,249,129,218
43,P19338,NCL,Nucleolin,AAAAAPASEDEDDEDDEDDEDDDDDEEDDSEEEAMETTPAK,AAAAAPASEDEDDEDDEDDEDDDDDEEDDS[79.9663]EEEAMET...,177,217,41,3,30S(79.9663),...,77,252,166,236,266,58,45,272,70,110
44,P19338,NCL,Nucleolin,AAAAAPASEDEDDEDDEDDEDDDDDEEDDSEEEAMETTPAK,AAAAAPASEDEDDEDDEDDEDDDDDEEDDS[79.9663]EEEAM[1...,177,217,41,3,"30S(79.9663),35M(15.9949)",...,77,252,166,236,266,58,45,272,70,110
45,P19338,NCL,Nucleolin,AAAAAPASEDEDDEDDEDDEDDDDDEEDDSEEEAMETTPAK,AAAAAPAS[79.9663]EDEDDEDDEDDEDDDDDEEDDSEEEAMET...,177,217,41,3,8S(79.9663),...,136,118,68,197,200,100,62,294,141,236
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171186,Q56P03,EAPP,E2F-associated phosphoprotein,YYDDIYFDSDSEDEDRAVQVTK,YYDDIYFDS[79.9663]DS[79.9663]EDEDRAVQVTK,101,122,22,"2,3","11S(79.9663),9S(79.9663)",...,79,207,177,219,262,103,38,267,47,154
171211,P35658,NUP214,Nuclear pore complex protein Nup214,YYEDLDEVSSTSSVSQSLESEDAR,YYEDLDEVS[79.9663]STSSVSQSLESEDAR,977,1000,24,2,9S(79.9663),...,76,178,140,147,188,66,60,251,81,121
171243,P40818,USP8,Ubiquitin carboxyl-terminal hydrolase 8,YYHSPTNTVHMYPPEMAPSSAPPSTPPTHK,YYHSPTNTVHMYPPEMAPSSAPPS[79.9663]TPPTHK,668,697,30,"3,4,5",24S(79.9663),...,224,165,115,234,241,236,116,272,123,247
171281,Q15029,EFTUD2,116 kDa U5 small nuclear ribonucleoprotein com...,YYPTAEEVYGPEVETIVQEEDTQPLTEPIIKPVK,YYPTAEEVYGPEVETIVQEEDT[79.9663]QPLTEPIIKPVK,65,98,34,"3,4",22T(79.9663),...,34,275,211,269,250,117,91,171,73,67


In [7]:
kinase_cols = ranked.columns[140:]
ranked["top_kinase"] = ranked[kinase_cols].idxmin(axis=1)
ranked["top_kinase"]

17        PRKD1
38         P38D
43        CK2A2
44        CK2A2
45        CK2A1
          ...  
171186    CK2A1
171211    CK1G2
171243    HIPK4
171281      ATM
171307    CK2A1
Name: top_kinase, Length: 8017, dtype: object

In [8]:
kinase_counts = ranked["top_kinase"].value_counts()
kinase_counts
# plt.figure(figsize=(14, 6))
# kinase_counts.plot(kind="bar")
# plt.xlabel("Kinase")
# plt.ylabel("Count")
# plt.title("Top kinase assignments per phosphosite")
# plt.xticks(rotation=45, ha="right")
# plt.tight_layout()
# plt.savefig("top_kinase_histogram.png", dpi=150)
# plt.show()

top_kinase
CK2A1    642
CK2A2    583
P38D     471
KIS      341
GSK3A    297
        ... 
NUAK2      1
HIPK1      1
MYLK4      1
ZAK        1
DLK        1
Name: count, Length: 253, dtype: int64